In [1]:
import numpy as np, pandas as pd

lg_seed = pd.read_csv("seed_results_lightgbm.csv")
ft_seed = pd.read_csv("seed_results_ft_transformer.csv")
lg_fold = pd.read_csv("fold_results_lightgbm_seeds.csv")
ft_fold = pd.read_csv("fold_results_ft_transformer_seeds.csv")

# FT-Transformer seed 42 was run earlier: numbers from your experiments.jsonl (lr 3e-4, all folds).
ft42_seed = dict(model="ft_transformer", seed=42, fold_mean_roc=0.875702, fold_mean_pr=0.575152,
                 pooled_roc=0.87485, pooled_pr=0.57274, brier=0.08773, logloss=0.28189, mean_best_iter=19.8)
ft42_fold = pd.DataFrame({"seed": 42, "fold": [0, 1, 2, 3, 4], "best_epoch": [22, 18, 22, 20, 17],
                          "roc_auc": [0.88003, 0.87310, 0.87684, 0.87726, 0.87128],
                          "pr_auc": [0.58558, 0.56280, 0.57927, 0.58509, 0.56302]})
if 42 not in ft_seed.seed.values:
    ft_seed = pd.concat([pd.DataFrame([ft42_seed]), ft_seed], ignore_index=True)
    ft_fold = pd.concat([ft42_fold, ft_fold], ignore_index=True)

print(f"Seeds used: LightGBM {sorted(lg_seed.seed)} | FT-Transformer {sorted(ft_seed.seed)}\n")

# 1) How much does each model move with the seed alone?
for metric, label in [("fold_mean_roc", "ROC-AUC (mean over folds)"), ("fold_mean_pr", "PR-AUC (mean over folds)")]:
    print(label)
    for name, d in [("LightGBM", lg_seed), ("FT-Transformer", ft_seed)]:
        v = d[metric].values
        print(f"  {name:15s} mean {v.mean():.5f} | seed std {v.std(ddof=1):.5f} | range {v.min():.5f} to {v.max():.5f}")
    a, b = ft_seed[metric].values, lg_seed[metric].values
    gap, noise = a.mean() - b.mean(), max(a.std(ddof=1), b.std(ddof=1))
    print(f"  gap (FT - LightGBM) {gap:+.5f} = {gap / noise:.1f}x the larger seed std | ranges overlap: {not (a.min() > b.max() or b.min() > a.max())}\n")

# 2) Fold-by-fold paired gap for EVERY pair of (FT seed, LightGBM seed)
rows = []
for sf in ft_fold.seed.unique():
    for sl in lg_fold.seed.unique():
        m = ft_fold[ft_fold.seed == sf].merge(lg_fold[lg_fold.seed == sl], on="fold", suffixes=("_ft", "_lg"))
        rows.append(dict(ft_seed=sf, lgbm_seed=sl, d_roc=(m.roc_auc_ft - m.roc_auc_lg).mean(),
                         d_pr=(m.pr_auc_ft - m.pr_auc_lg).mean(), folds_ft_ahead=int((m.roc_auc_ft > m.roc_auc_lg).sum()), n_folds=len(m)))
g = pd.DataFrame(rows)
print(f"Paired per-fold gap across {len(g)} seed pairs:")
print(f"  ROC-AUC gap: mean {g.d_roc.mean():+.5f} | min {g.d_roc.min():+.5f} | max {g.d_roc.max():+.5f} | FT ahead in {(g.d_roc > 0).sum()} of {len(g)} pairs")
print(f"  PR-AUC  gap: mean {g.d_pr.mean():+.5f} | min {g.d_pr.min():+.5f} | max {g.d_pr.max():+.5f} | FT ahead in {(g.d_pr > 0).sum()} of {len(g)} pairs")

Seeds used: LightGBM [42, 43, 44, 45, 46] | FT-Transformer [42, 43, 44]

ROC-AUC (mean over folds)
  LightGBM        mean 0.87406 | seed std 0.00012 | range 0.87389 to 0.87419
  FT-Transformer  mean 0.87544 | seed std 0.00028 | range 0.87515 to 0.87570
  gap (FT - LightGBM) +0.00138 = 5.0x the larger seed std | ranges overlap: False

PR-AUC (mean over folds)
  LightGBM        mean 0.57198 | seed std 0.00031 | range 0.57160 to 0.57231
  FT-Transformer  mean 0.57473 | seed std 0.00054 | range 0.57412 to 0.57515
  gap (FT - LightGBM) +0.00275 = 5.1x the larger seed std | ranges overlap: False

Paired per-fold gap across 15 seed pairs:
  ROC-AUC gap: mean +0.00138 | min +0.00096 | max +0.00182 | FT ahead in 15 of 15 pairs
  PR-AUC  gap: mean +0.00275 | min +0.00182 | max +0.00355 | FT ahead in 15 of 15 pairs
